## 📖 Background
You work for an international HR consultancy helping companies attract and retain top talent in the competitive tech industry. As part of your services, you provide clients with insights into industry salary trends to ensure they remain competitive in hiring and compensation practices.

Your team wants to use a data-driven approach to analyse how various factors—such as job role, experience level, remote work, and company size—impact salaries globally. By understanding these trends, you can advise clients on offering competitive packages to attract the best talent.

In this competition, you’ll explore and visualise salary data from thousands of employees worldwide. f you're tackling the advanced level, you'll go a step further—building predictive models to uncover key salary drivers and providing insights on how to enhance future data collection.

## 💾 The data

The data comes from a survey hosted by an HR consultancy, available in `'salaries.csv'`.

#### Each row represents a single employee's salary record for a given year:
- **`work_year`** - The year the salary was paid.  
- **`experience_level`** - Employee experience level:  
  - **`EN`**: Entry-level / Junior  
  - **`MI`**: Mid-level / Intermediate  
  - **`SE`**: Senior / Expert  
  - **`EX`**: Executive / Director  
- **`employment_type`** - Employment type:  
  - **`PT`**: Part-time  
  - **`FT`**: Full-time  
  - **`CT`**: Contract  
  - **`FL`**: Freelance  
- **`job_title`** - The job title during the year.  
- **`salary`** - Gross salary paid (in local currency).  
- **`salary_currency`** - Salary currency (ISO 4217 code).  
- **`salary_in_usd`** - Salary converted to USD using average yearly FX rate.  
- **`employee_residence`** - Employee's primary country of residence (ISO 3166 code).  
- **`remote_ratio`** - Percentage of remote work:  
  - **`0`**: No remote work (<20%)  
  - **`50`**: Hybrid (50%)  
  - **`100`**: Fully remote (>80%)  
- **`company_location`** - Employer's main office location (ISO 3166 code).  
- **`company_size`** - Company size:  
  - **`S`**: Small (<50 employees)  
  - **`M`**: Medium (50–250 employees)  
  - **`L`**: Large (>250 employees)  

In [1]:
import pandas as pd
salaries_df = pd.read_csv('salaries.csv')
salaries_df

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2024,MI,FT,Developer,168276,USD,168276,US,0,US,M
1,2024,MI,FT,Developer,112184,USD,112184,US,0,US,M
2,2024,EN,FT,Developer,180000,USD,180000,US,0,US,M
3,2024,EN,FT,Developer,133500,USD,133500,US,0,US,M
4,2024,EN,FT,Developer,122000,USD,122000,US,0,US,M
...,...,...,...,...,...,...,...,...,...,...,...
57189,2020,SE,FT,Data Scientist,412000,USD,412000,US,100,US,L
57190,2021,MI,FT,Principal Data Scientist,151000,USD,151000,US,100,US,L
57191,2020,EN,FT,Data Scientist,105000,USD,105000,US,100,US,S
57192,2020,EN,CT,Business Data Analyst,100000,USD,100000,US,100,US,L


In [2]:
SELECT COUNT(*) AS total_rows
FROM 'salaries.csv';

,total_rows
0,57194


In [3]:
SELECT max(work_year), min(work_year), max(work_year) - min(work_year) AS range_covered
FROM 'salaries.csv';

,max(work_year),min(work_year),range_covered
0,2024,2020,4


In [4]:
SELECT job_title, ROUND(AVG(salary_in_usd), 2) AS avg_salary
FROM 'salaries.csv'
WHERE job_title IN ('Data Engineer', 'Data Scientist')
GROUP BY 1

,job_title,avg_salary
0,Data Engineer,149315.00
1,Data Scientist,159397.07


In [8]:
WITH RankSalarios AS (
    SELECT 
        work_year,
        job_title,
        ROUND(AVG(salary_in_usd), 2) AS avg_salary,
        ROW_NUMBER() OVER(PARTITION BY work_year ORDER BY AVG(salary_in_usd) DESC) AS rank_ano
    FROM 'salaries.csv'
    WHERE job_title IN ('Data Engineer', 'Data Scientist')
    GROUP BY work_year, job_title
)
SELECT 
    work_year,
    job_title AS profissao_no_topo,
    avg_salary AS salario_medio
FROM RankSalarios
WHERE rank_ano = 1;

,work_year,profissao_no_topo,salario_medio
0,2024,Data Scientist,160158.09
1,2021,Data Engineer,96143.08
2,2020,Data Scientist,108943.24
3,2022,Data Scientist,139644.64
4,2023,Data Scientist,163855.09


In [14]:
SELECT 
    company_location AS pais_da_empresa,
    COUNT(*) AS total_estrangeiros_contratados,
    ROUND(AVG(salary_in_usd), 2) AS media_salario_estrangeiro,
    -- Compara com a média geral do país para ver se a empresa economiza contratando fora
    ROUND(AVG(salary_in_usd) - AVG(AVG(salary_in_usd)) OVER(PARTITION BY company_location), 2) AS desvio_da_media_local
FROM 'salaries.csv'
-- Filtra apenas onde o funcionário mora em país diferente da empresa
WHERE employee_residence <> company_location 
GROUP BY company_location
HAVING COUNT(*) > 5 -- Filtra países com amostragem relevante
ORDER BY total_estrangeiros_contratados DESC
LIMIT 10;

,pais_da_empresa,total_estrangeiros_contratados,media_salario_estrangeiro,desvio_da_media_local
0,US,59,87821.85,0.0
1,DE,13,55819.00,0.0
2,GB,9,67029.33,0.0
3,CA,6,81370.33,0.0
4,AU,6,66835.33,0.0


In [13]:
SELECT 
    CASE 
        WHEN remote_ratio = 0 THEN 'On-site'
        WHEN remote_ratio = 50 THEN 'Hybrid'
        WHEN remote_ratio = 100 THEN 'Full remote'
        ELSE 'Outro'
    END AS work_type,
    COUNT(*) AS total_employees,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS total_percentage,
    ROUND(AVG(salary_in_usd), 2) AS avg_salary_usd
FROM 'salaries.csv'
WHERE employment_type = 'FT'
GROUP BY remote_ratio
ORDER BY avg_salary_usd DESC;

,work_type,total_employees,total_percentage,avg_salary_usd
0,On-site,44225,77.62,162585.20
1,Full remote,12493,21.93,150095.22
2,Hybrid,257,0.45,83087.39


### How many records are in the dataset?
- There are 57,194 rows in this dataset.
### What is the range of years covered?
- The range covered in this dataset is 4 years, going from 2020 to 2024.
### What is the average salary(in USD) for Data Scientists and Data Engineers?
- Average salary for Data Scientist: $159,397.07
- Average salary for Data Engineer: $149,315.00
### Which role earns more on average?
- Data Scientists earn more on average.
### How many full-time employees based in the US work 100% remotely?
- 11,125 employees are based in the US and work 100% remotely.

## 💪 Competition challenge

In this first level, you’ll explore and summarise the dataset to understand its structure and key statistics. If you want to push yourself further, check out level two!
Create a report that answers the following:
- How many records are in the dataset, and what is the range of years covered?
- What is the average salary (in USD) for Data Scientists and Data Engineers? Which role earns more on average?
- How many full-time employees based in the US work 100% remotely?

## 🧑‍⚖️ Judging criteria

This is a community-based competition. Once the competition concludes, you'll have the opportunity to view and vote for the best submissions of others as the voting begins. The top 5 most upvoted entries will win. The winners will receive DataCamp merchandise.

## ✅ Checklist before publishing into the competition
- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as notebook.ipynb.
- **Remove redundant cells** like the judging criteria, so the workbook is focused on your story.
- Make sure the workbook reads well and explains how you found your insights. 
- Try to include an **executive summary** of your recommendations at the beginning.
- Check that all the cells run without error

## ⌛️ Time is ticking. Good luck!